# Text Preprocessing - Information Retrieval
**Topik:** Manajemen Energi  
**Pipeline:** Case Folding → Tokenisasi → Stop-word Removal → Stemming → Forward Index → Inverted Index → Boolean Retrieval

## 1. Import Library

In [7]:
pip install PySastrawi


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import re
import pandas as pd
from collections import defaultdict
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

## 2. Load Data dari CSV

In [9]:
import pandas as pd
import os

# Read CSV file
df = pd.read_csv('TKI_Keyword_Manajemen Energi(Jurnal).csv')
print(f'File "Manajemen_Energi.csv" loaded successfully.')

File "Manajemen_Energi.csv" loaded successfully.


In [10]:
import pandas as pd
import re

def load_sentences(filepath):
    """
    Membaca file CSV dari filepath dan mengekstrak kalimat dari kolom pertama.
    Baris yang merupakan URL atau baris kosong dilewati.
    Tanda kutip pembungkus dan spasi tepi dibersihkan.
    """
    # Read the CSV file
    df = pd.read_csv(filepath, encoding='utf-8-sig')
    sentences = []

    # Ambil kolom pertama
    for index, row in df.iterrows():
        line = str(row.iloc[0]).strip()
        if not line or pd.isna(line) or line == 'nan': # Skip empty or NaN lines
            continue
        
        # Skip URL
        if line.startswith('http'):
            continue

        # Menghapus tanda kutip pembungkus jika ada
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1]
        
        line = line.strip()
        if len(line) > 10:
            sentences.append(line)

    return sentences


filepath = 'TKI_Keyword_Manajemen Energi(Jurnal).csv'
sentences = load_sentences(filepath)

print(f'Total kalimat dimuat: {len(sentences)}')
print()
for i, s in enumerate(sentences[:3], 1):
    print(f'[{i}] {s[:90]}...')

Total kalimat dimuat: 58

[1] Tujuan dari penelitian ini adalah untuk mengeksplorasi dampak Akuntansi hijau terhadap kin...
[2] Penelitian sebelumnya menunjukkan bahwa ketika efisiensi energi dianggap strategis, perusa...
[3] Aspek lain dari manajemen energi telah berkembang dari efisiensi biaya murni menjadi efisi...


## 3. Definisi Stop-word

In [11]:
STOPWORDS = {
    # Kata tugas umum
    'yang', 'adalah', 'ini', 'itu', 'untuk', 'dari', 'ke', 'di', 'pada', 'dengan',
    'tidak', 'juga', 'telah', 'oleh', 'dan', 'atau', 'dalam', 'bahwa', 'dapat',
    'akan', 'lebih', 'sudah', 'harus', 'ada', 'saja', 'jika', 'agar', 'atas',
    'bagi', 'antara', 'namun', 'maka', 'karena', 'sehingga', 'secara', 'seperti',
    'sesuai', 'masih', 'belum', 'bisa', 'semua', 'setiap', 'sangat', 'serta',
    'melalui', 'tetapi', 'hingga', 'saat', 'ketika', 'setelah', 'sebelum',
    'selama', 'selain', 'memiliki', 'menjadi', 'sebagai', 'suatu', 'sebuah',
    'hal', 'cara', 'lain', 'maupun', 'merupakan', 'mengenai', 'tersebut',
    'beberapa', 'berbagai', 'terhadap', 'kepada', 'tentang', 'salah', 'satu',
    'jadi', 'bukan', 'terdapat', 'berdasarkan', 'menurut', 'paling', 'hanya',
    'bahkan', 'hampir', 'pula', 'pasti', 'tentu', 'sering', 'selalu',
    # Kata ganti
    'ia', 'dia', 'mereka', 'kami', 'kita', 'saya', 'anda', 'kamu',
    # Preposisi dan konjungsi tambahan
    'dimana', 'per', 'masing', 'sendiri', 'tiap', 'sekaligus', 'kemudian',
    'selanjutnya', 'akhirnya', 'terkait', 'berkaitan', 'terus',
    # Noise domain
    'iot', 'sni', 'kwh', 'rsud', 'smk', 'umkm', 'gambar', 'tabel',
    'google', 'pubmed', 'sciencedirect', 'garuda', 'scholar',
    'internet', 'things', 'al', 'et', 'no',
    # Kata kerja generik
    'menunjukkan', 'menyatakan', 'menjelaskan', 'membahas', 'mengkaji',
    'menelusuri', 'menganalisis', 'merangkum', 'menyimpulkan',
    'dilakukan', 'dilaksanakan', 'diterapkan', 'digunakan', 'diketahui',
    'ditemukan', 'didapatkan', 'diperoleh', 'dikembangkan',
    'bertujuan', 'berfungsi', 'berupa', 'bersifat',
}

print(f'Total stop-word: {len(STOPWORDS)}')

Total stop-word: 146


## 4. Inisialisasi Stemmer (PySastrawi)

In [12]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Uji stemmer
contoh = ['manajemen', 'penerapan', 'meningkatkan', 'berkelanjutan', 'memediasi']
for kata in contoh:
    print(f'{kata:20s} -> {stemmer.stem(kata)}')

manajemen            -> manajemen
penerapan            -> terap
meningkatkan         -> tingkat
berkelanjutan        -> lanjut
memediasi            -> mediasi


## 5. Fungsi Preprocessing

In [13]:
def case_folding(text):
    """
    Mengubah teks menjadi huruf kecil dan menghapus
    tanda baca, angka, serta karakter non-alfabet.
    """
    text = text.lower()
    text = re.sub(r'\(.*?\)', ' ', text)   # Menghapus isi dalam kurung
    text = re.sub(r'[^a-z\s]', ' ', text)  # Menghapus non-huruf
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenisasi(text):
    """
    Memecah teks menjadi daftar token kata tunggal.
    Token dengan panjang kurang dari 3 karakter dibuang.
    """
    return [t for t in text.split() if len(t) >= 3]


def hapus_stopword(tokens):
    """
    Membuang token yang termasuk dalam daftar stop-word.
    """
    return [t for t in tokens if t not in STOPWORDS]


def stemming(tokens):
    """
    Mengembalikan setiap token ke bentuk kata dasarnya
    menggunakan algoritma Nazief-Adriani via PySastrawi.
    Hasil yang masih termasuk stop-word setelah stemming dibuang.
    """
    hasil = []
    for t in tokens:
        stem = stemmer.stem(t)
        if stem not in STOPWORDS and len(stem) >= 3:
            hasil.append(stem)
    return hasil


def preprocessing(doc_id, raw_text):
    """
    Menjalankan seluruh pipeline preprocessing pada satu kalimat.
    Mengembalikan dictionary dengan setiap tahap sebagai key.
    """
    folded   = case_folding(raw_text)
    tokens   = tokenisasi(folded)
    no_stop  = hapus_stopword(tokens)
    stemmed  = stemming(no_stop)

    return {
        'DocID'             : doc_id,
        'Teks Mentah'       : raw_text.strip(),
        'Case Folding'      : folded,
        'Tokenisasi'        : tokens,
        'Stopword Removal'  : no_stop,
        'Stemming'          : stemmed,
    }

## 6. Preprocessing Seluruh Dokumen

In [14]:
records = []
for i, kalimat in enumerate(sentences, 1):
    record = preprocessing(f'Doc {i}', kalimat)
    records.append(record)

print(f'Total dokumen diproses: {len(records)}')

Total dokumen diproses: 58


## 7. Tabel Preprocessing Lengkap

In [15]:
df_prep = pd.DataFrame(records)

# Kolom tampilan: ubah list menjadi string untuk tabel
df_display = df_prep.copy()
df_display['Tokenisasi']       = df_display['Tokenisasi'].apply(lambda x: ', '.join(x))
df_display['Stopword Removal'] = df_display['Stopword Removal'].apply(lambda x: ', '.join(x))
df_display['Stemming']         = df_display['Stemming'].apply(lambda x: ', '.join(x))

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', len(df_display))

df_display[['DocID', 'Teks Mentah', 'Case Folding', 'Tokenisasi', 'Stopword Removal', 'Stemming']]

,DocID,Teks Mentah,Case Folding,Tokenisasi,Stopword Removal,Stemming
0,Doc 1,Tujuan dari penelitian ini adalah untuk mengeksplorasi d...,tujuan dari penelitian ini adalah untuk mengeksplorasi d...,"tujuan, dari, penelitian, ini, adalah, untuk, mengeksplo...","tujuan, penelitian, mengeksplorasi, dampak, akuntansi, h...","tuju, teliti, eksplorasi, dampak, akuntansi, hijau, kerj..."
1,Doc 2,Penelitian sebelumnya menunjukkan bahwa ketika efisiensi...,penelitian sebelumnya menunjukkan bahwa ketika efisiensi...,"penelitian, sebelumnya, menunjukkan, bahwa, ketika, efis...","penelitian, sebelumnya, efisiensi, energi, dianggap, str...","teliti, efisiensi, energi, anggap, strategis, usaha, cen..."
2,Doc 3,Aspek lain dari manajemen energi telah berkembang dari e...,aspek lain dari manajemen energi telah berkembang dari e...,"aspek, lain, dari, manajemen, energi, telah, berkembang,...","aspek, manajemen, energi, berkembang, efisiensi, biaya, ...","aspek, manajemen, energi, kembang, efisiensi, biaya, mur..."
3,Doc 4,"Di sisi lain, diketahui bahwa sistem manajemen energi ma...",di sisi lain diketahui bahwa sistem manajemen energi mam...,"sisi, lain, diketahui, bahwa, sistem, manajemen, energi,...","sisi, sistem, manajemen, energi, mampu, menjamin, kualit...","sisi, sistem, manajemen, energi, mampu, jamin, kualitas,..."
4,Doc 5,Hal ini menunjukkan bahwa sistem manajemen energi memban...,hal ini menunjukkan bahwa sistem manajemen energi memban...,"hal, ini, menunjukkan, bahwa, sistem, manajemen, energi,...","sistem, manajemen, energi, membantu, menyediakan, lingku...","sistem, manajemen, energi, bantu, sedia, lingkung, ruang..."
5,Doc 6,Pengaruh antara akuntansi hijau dan kinerja berkelanjuta...,pengaruh antara akuntansi hijau dan kinerja berkelanjuta...,"pengaruh, antara, akuntansi, hijau, dan, kinerja, berkel...","pengaruh, akuntansi, hijau, kinerja, berkelanjutan, dime...","pengaruh, akuntansi, hijau, kerja, lanjut, mediasi, mana..."
6,Doc 7,Hasil penelitian ini sejalan dengan temuan Rahman dan Is...,hasil penelitian ini sejalan dengan temuan rahman dan is...,"hasil, penelitian, ini, sejalan, dengan, temuan, rahman,...","hasil, penelitian, sejalan, temuan, rahman, islam, manaj...","hasil, teliti, jalan, temu, rahman, islam, manajemen, en..."
7,Doc 8,Manajemen energi dapat memediasi pengaruh akuntansi hija...,manajemen energi dapat memediasi pengaruh akuntansi hija...,"manajemen, energi, dapat, memediasi, pengaruh, akuntansi...","manajemen, energi, memediasi, pengaruh, akuntansi, hijau...","manajemen, energi, mediasi, pengaruh, akuntansi, hijau, ..."
8,Doc 9,Hasil penelitian ini menunjukkan bahwa penerapan akuntan...,hasil penelitian ini menunjukkan bahwa penerapan akuntan...,"hasil, penelitian, ini, menunjukkan, bahwa, penerapan, a...","hasil, penelitian, penerapan, akuntansi, hijau, manajeme...","hasil, teliti, terap, akuntansi, hijau, manajemen, energ..."
9,Doc 10,"Di sisi lain, diketahui bahwa sistem manajemen energi ma...",di sisi lain diketahui bahwa sistem manajemen energi mam...,"sisi, lain, diketahui, bahwa, sistem, manajemen, energi,...","sisi, sistem, manajemen, energi, mampu, menjamin, kualit...","sisi, sistem, manajemen, energi, mampu, jamin, kualitas,..."


## 8. Forward Index

In [16]:
def buat_forward_index(records):
    """
    Membangun forward index: DocID -> {term: frekuensi}.
    Menghitung frekuensi kemunculan setiap term dalam satu dokumen.
    """
    forward_index = {}
    for rec in records:
        freq = defaultdict(int)
        for term in rec['Stemming']:
            freq[term] += 1
        forward_index[rec['DocID']] = dict(sorted(freq.items()))
    return forward_index


forward_index = buat_forward_index(records)

# Tampilkan sebagai tabel
fi_rows = []
for doc_id, terms in forward_index.items():
    term_freq_str = ', '.join([f'{t}:{f}' for t, f in sorted(terms.items(), key=lambda x: -x[1])])
    fi_rows.append({
        'DocID'              : doc_id,
        'Term : Frekuensi'   : term_freq_str,
        'Jumlah Unique Term' : len(terms),
    })

df_forward = pd.DataFrame(fi_rows)
df_forward

,DocID,Term : Frekuensi,Jumlah Unique Term
0,Doc 1,"akuntansi:1, dampak:1, eksplorasi:1, energi:1, hijau:1, ...",11
1,Doc 2,"energi:2, anggap:1, cenderung:1, efisiensi:1, manajemen:...",10
2,Doc 3,"efisiensi:3, aspek:1, biaya:1, energi:1, kembang:1, kerj...",14
3,Doc 4,"energi:2, beri:1, efisiensi:1, jamin:1, kualitas:1, ling...",11
4,Doc 5,"bantu:1, energi:1, karyawan:1, kualitas:1, langgan:1, li...",12
5,Doc 6,"akuntansi:1, energi:1, hijau:1, kerja:1, lanjut:1, manaj...",8
6,Doc 7,"akuntansi:1, energi:1, hasil:1, hijau:1, islam:1, jalan:...",14
7,Doc 8,"akuntansi:1, energi:1, hijau:1, kerja:1, lingkung:1, man...",8
8,Doc 9,"akuntansi:1, bisnis:1, energi:1, hasil:1, hijau:1, langs...",9
9,Doc 10,"energi:2, beri:1, efisiensi:1, jamin:1, kualitas:1, ling...",11


## 9. Inverted Index

In [17]:
def buat_inverted_index(records):
    """
    Membangun inverted index: term -> [DocID, ...].
    Setiap term dipetakan ke daftar dokumen yang mengandungnya.
    """
    inverted = defaultdict(list)
    for rec in records:
        for term in set(rec['Stemming']):
            inverted[term].append(rec['DocID'])

    # Urutkan dictionary dan posting list
    inverted_sorted = {}
    for term in sorted(inverted.keys()):
        posting = sorted(inverted[term], key=lambda x: int(x.split()[1]))
        inverted_sorted[term] = posting

    return inverted_sorted


inverted_index = buat_inverted_index(records)

# Tampilkan sebagai tabel
ii_rows = []
for i, (term, posting) in enumerate(inverted_index.items(), 1):
    ii_rows.append({
        'No'          : i,
        'Kata Dasar'  : term,
        'Posting List': ', '.join(posting),
        'df'          : len(posting),
    })

df_inverted = pd.DataFrame(ii_rows)
print(f'Total term unik: {len(df_inverted)}')
df_inverted

Total term unik: 268


,No,Kata Dasar,Posting List,df
0,1,abdi,Doc 21,1
1,2,agenda,Doc 49,1
2,3,aktif,Doc 49,1
3,4,akuntansi,"Doc 1, Doc 6, Doc 7, Doc 8, Doc 9, Doc 12",6
4,5,alat,"Doc 15, Doc 28, Doc 48, Doc 50, Doc 54, Doc 55, Doc 57",7
...,...,...,...,...
263,264,usaha,Doc 2,1
264,265,utama,"Doc 16, Doc 19, Doc 20",3
265,266,waktu,"Doc 28, Doc 36, Doc 37",3
266,267,wawas,Doc 48,1


## 10. Boolean Retrieval

In [18]:
def get_posting(term):
    """
    Mengambil posting list untuk satu term.
    Term di-stem terlebih dahulu sebelum dicari di index.
    """
    key = stemmer.stem(term.lower())
    return set(inverted_index.get(key, []))


# Query 1: ("manajemen" AND "energi") AND "efisiensi"
p_manajemen = get_posting('manajemen')
p_energi    = get_posting('energi')
p_efisiensi = get_posting('efisiensi')

hasil_q1 = p_manajemen & p_energi & p_efisiensi

# Query 2: "konservasi" OR "efisiensi"
p_konservasi = get_posting('konservasi')

hasil_q2 = p_konservasi | p_efisiensi


def sort_doc(doc_set):
    return sorted(doc_set, key=lambda x: int(x.split()[1]))


# Tampilkan hasil
br_rows = [
    {
        'Query'    : '("manajemen" AND "energi") AND "efisiensi"',
        'Operasi'  : 'AND (interseksi)',
        'Hasil'    : ', '.join(sort_doc(hasil_q1)) if hasil_q1 else 'Tidak ada hasil',
        'Jumlah'   : len(hasil_q1),
    },
    {
        'Query'    : '"konservasi" OR "efisiensi"',
        'Operasi'  : 'OR (gabungan)',
        'Hasil'    : ', '.join(sort_doc(hasil_q2)),
        'Jumlah'   : len(hasil_q2),
    },
]

df_boolean = pd.DataFrame(br_rows)
df_boolean

,Query,Operasi,Hasil,Jumlah
0,"(""manajemen"" AND ""energi"") AND ""efisiensi""",AND (interseksi),"Doc 2, Doc 3, Doc 4, Doc 10, Doc 25, Doc 27, Doc 32, Doc...",13
1,"""konservasi"" OR ""efisiensi""",OR (gabungan),"Doc 2, Doc 3, Doc 4, Doc 10, Doc 15, Doc 25, Doc 27, Doc...",14


## 11. Incidence Matrix (TF-Biner)

In [19]:
query_terms = ['manajemen', 'energi', 'efisiensi', 'konservasi', 'transisi']
query_stems = [stemmer.stem(t) for t in query_terms]

matrix_rows = []
for rec in records:
    row = {'DocID': rec['DocID']}
    for term_raw, term_stem in zip(query_terms, query_stems):
        row[term_raw] = 1 if term_stem in rec['Stemming'] else 0
    matrix_rows.append(row)

df_matrix = pd.DataFrame(matrix_rows).set_index('DocID')

print('Incidence Matrix (TF-Biner) - 5 term kunci')
print(f'Total dokumen: {len(df_matrix)}')
df_matrix

Incidence Matrix (TF-Biner) - 5 term kunci
Total dokumen: 58


,manajemen,energi,efisiensi,konservasi,transisi
DocID,,,,,
Doc 1,1,1,0,0,0
Doc 2,1,1,1,0,0
Doc 3,1,1,1,0,1
Doc 4,1,1,1,0,0
Doc 5,1,1,0,0,0
Doc 6,1,1,0,0,0
Doc 7,1,1,0,0,0
Doc 8,1,1,0,0,0
Doc 9,1,1,0,0,0


## 12. Simpan Semua Tabel ke Excel

In [20]:
output_file = 'IR_ManajemenEnergi_Output.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

    df_display[['DocID', 'Teks Mentah', 'Case Folding',
                'Tokenisasi', 'Stopword Removal', 'Stemming']].to_excel(
        writer, sheet_name='Preprocessing', index=False
    )

    df_forward.to_excel(
        writer, sheet_name='Forward Index', index=False
    )

    df_inverted.to_excel(
        writer, sheet_name='Inverted Index', index=False
    )

    df_boolean.to_excel(
        writer, sheet_name='Boolean Retrieval', index=False
    )

    df_matrix.to_excel(
        writer, sheet_name='Incidence Matrix'
    )

print(f'File disimpan: {output_file}')

File disimpan: IR_ManajemenEnergi_Output.xlsx


## 13. TF-IDF dan Vector Space Model (VSM)



In [21]:
import math

def calculate_tf_idf(records, inverted_index):
    """
    Menghitung bobot TF-IDF untuk setiap term di setiap dokumen.
    """
    tf_idf_scores = defaultdict(dict)
    num_documents = len(records)

    # Menghitung IDF (Inverse Document Frequency)
    idf_scores = {}
    for term, posting_list in inverted_index.items():
        df = len(posting_list) # Frekuensi Dokumen
        idf_scores[term] = math.log10(num_documents / (df + 1))

    # Menghitung TF-IDF
    for record in records:
        doc_id = record['DocID']
        stemmed_terms = record['Stemming']
        term_freq = defaultdict(int)
        for term in stemmed_terms:
            term_freq[term] += 1

        for term in set(stemmed_terms):
            tf = term_freq[term] / len(stemmed_terms)
            idf = idf_scores.get(term, 0)
            tf_idf_scores[doc_id][term] = tf * idf

    return tf_idf_scores, idf_scores

tf_idf_doc_scores, idf_scores = calculate_tf_idf(records, inverted_index)

print("Contoh TF-IDF untuk Doc 1:")
for term, score in list(tf_idf_doc_scores['Doc 1'].items())[:5]: # Menampilkan 5 term pertama dari Doc 1
    print(f"  {term}: {score:.4f}")

# Menyiapkan data untuk tampilan DataFrame
tfidf_rows = []
for doc_id, terms in tf_idf_doc_scores.items():
    sorted_terms = sorted(terms.items(), key=lambda item: item[1], reverse=True)
    tfidf_rows.append({
        'DocID': doc_id,
        'TF-IDF Scores': ', '.join([f'{t}:{s:.4f}' for t, s in sorted_terms])
    })

df_tfidf = pd.DataFrame(tfidf_rows)
df_tfidf.head()

Contoh TF-IDF untuk Doc 1:
  teliti: 0.0694
  akuntansi: 0.0835
  manajemen: -0.0007
  energi: -0.0007
  lingkung: 0.0694


,DocID,TF-IDF Scores
0,Doc 1,"mediator:0.1329, dampak:0.1329, eksplorasi:0.1329, tuju:..."
1,Doc 2,"tinggi:0.1329, cenderung:0.1329, anggap:0.1329, usaha:0...."
2,Doc 3,"efisiensi:0.1157, murni:0.0914, transisi:0.0914, sedang:..."
3,Doc 4,"jamin:0.1072, sisi:0.0968, beri:0.0968, kualitas:0.0821,..."
4,Doc 5,"bantu:0.1072, karyawan:0.1072, sedia:0.1072, langgan:0.1..."


In [22]:
def calculate_cosine_similarity(query_vector, document_vector):
    """
    Menghitung kesamaan kosinus antara dua vektor.
    """
    dot_product = sum(query_vector.get(term, 0) * document_vector.get(term, 0) for term in set(query_vector) | set(document_vector))

    magnitude_query = math.sqrt(sum(q_val**2 for q_val in query_vector.values()))
    magnitude_document = math.sqrt(sum(d_val**2 for d_val in document_vector.values()))

    if magnitude_query == 0 or magnitude_document == 0:
        return 0
    return dot_product / (magnitude_query * magnitude_document)

def query_vsm(query_text, tf_idf_doc_scores, idf_scores, records, stemmer, stopwords):
    """
    Mencari dokumen berdasarkan kueri menggunakan VSM dan kesamaan kosinus.
    """
    # 1. Preprocessing kueri
    processed_query = preprocessing('Query', query_text)
    query_stemmed_terms = processed_query['Stemming']

    # 2. Menghitung TF untuk kueri
    query_tf = defaultdict(int)
    for term in query_stemmed_terms:
        query_tf[term] += 1

    # 3. Menghitung TF-IDF untuk kueri
    query_tf_idf = {}
    if len(query_stemmed_terms) > 0:
        for term in set(query_stemmed_terms):
            tf = query_tf[term] / len(query_stemmed_terms)
            idf = idf_scores.get(term, 0)
            query_tf_idf[term] = tf * idf

    # 4. Menghitung kesamaan kosinus antara kueri dan setiap dokumen
    similarities = {}
    for doc_id, doc_tf_idf_vector in tf_idf_doc_scores.items():
        similarity = calculate_cosine_similarity(query_tf_idf, doc_tf_idf_vector)
        if similarity > 0:
            similarities[doc_id] = similarity

    # 5. Mengurutkan dokumen berdasarkan kesamaan
    ranked_documents = sorted(similarities.items(), key=lambda item: item[1], reverse=True)

    return ranked_documents

query = "Kebutuhan energi saat ini meningkat pesat"
ranked_docs = query_vsm(query, tf_idf_doc_scores, idf_scores, records, stemmer, STOPWORDS)

print(f"Dokumen yang relevan untuk kueri '{query}':")
if ranked_docs:
    for doc_id, score in ranked_docs[:10]:
        original_text = next((rec['Teks Mentah'] for rec in records if rec['DocID'] == doc_id), 'N/A')
        print(f"  {doc_id} (Similarity: {score:.4f}): {original_text[:100]}...")
else:
    print("Tidak ada dokumen yang relevan ditemukan.")

Dokumen yang relevan untuk kueri 'Kebutuhan energi saat ini meningkat pesat':
  Doc 51 (Similarity: 0.2659): Dengan demikian, manajemen energi tidak hanya menjawab tantangan peningkatan kebutuhan listrik, teta...
  Doc 37 (Similarity: 0.2245): Data ini berfungsi sebagai indikator penting untuk melihat sejauh mana kebutuhan energi listrik masy...
  Doc 36 (Similarity: 0.2088): Dalam konteks ini, pemahaman terhadap konsumsi listrik per kapita menjadi sangat relevan untuk melih...
  Doc 43 (Similarity: 0.1062): Salah satu langkah awal dalam menerapkan manajemen energi adalah peningkatan literasi energi masyara...
  Doc 56 (Similarity: 0.1055): Efisiensi penggunaan energi masih mungkin ditingkatkan melalui penerapan sistem manajemen energi ter...
  Doc 53 (Similarity: 0.1055): Efisiensi penggunaan energimasih mungkin ditingkatkan melalui penerapan sistem manajemen energi....
  Doc 24 (Similarity: 0.0988): Kegiatan ini dapat meningkatkan kemampuan peserta dalam melakukan audit dan manajemen

In [23]:
output_file = 'IR_ManajemenEnergi_Output.xlsx'

# Result
df_vsm_results = pd.DataFrame(ranked_docs, columns=['DocID', 'Similarity Score'])

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_tfidf.to_excel(
        writer, sheet_name='TF-IDF Scores', index=False
    )
    df_vsm_results.to_excel(
        writer, sheet_name='VSM Query Results', index=False
    )

print(f'TF-IDF scores and VSM query results {output_file}')

TF-IDF scores and VSM query results IR_ManajemenEnergi_Output.xlsx
